# 🔬 Notebook 3 — Stock Exchange: Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Then **select the `.venv` kernel** in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses only the Python standard library + `pydantic` — nothing else to install, no Docker required.


## What we'll build

Four short deep-dives, each as a **bad → best** pair of runnable cells:

1. **Matching engine** — from a naive list scan to a price-level book with FIFO
   queues, then a test suite that actually *proves* price-time priority.
2. **Order types** — `limit`, `market`, `IOC`, `FOK`, and what each one does to
   the remainder that could not be filled.
3. **Determinism & replay** — how a WAL lets us rebuild state after a crash.
4. **Market-data fan-out** — why the matcher must **never** push to subscribers directly.

You can run each pair in isolation.

## 1. Matching engine — price-time priority

**Price-time priority**: at any given price level, the order that arrived **first** is matched first. It is the fairness rule that every major exchange uses.

### Bad version: scan a flat list

Works, but every incoming order does an O(n) scan over all resting orders. With 1 M resting orders this is unusable.

In [ ]:
from dataclasses import dataclass, field
from itertools import count
import time, random

@dataclass
class Order:
    id: int
    side: str           # 'buy' | 'sell'
    price: int          # integer ticks — see Notebook 2. Ignored for 'market'.
    qty: int
    ts: int = field(default_factory=count().__next__)   # arrival order
    otype: str = "limit"    # 'limit' | 'market' | 'ioc' | 'fok' — used in dive 2

def match_bad(resting: list[Order], incoming: Order):
    trades = []
    # BAD: linear scan, plus we must re-sort every time to find the best price
    opposite = "sell" if incoming.side == "buy" else "buy"
    candidates = [o for o in resting if o.side == opposite]
    # price-time priority: best price first, oldest first
    candidates.sort(key=lambda o: (o.price if incoming.side == "buy" else -o.price, o.ts))
    for o in candidates:
        if incoming.qty == 0: break
        crosses = (incoming.side == "buy"  and o.price <= incoming.price) or \
                  (incoming.side == "sell" and o.price >= incoming.price)
        if not crosses: break
        fill = min(o.qty, incoming.qty)
        trades.append((incoming.id, o.id, o.price, fill))
        o.qty -= fill; incoming.qty -= fill
    resting[:] = [o for o in resting if o.qty > 0]
    if incoming.qty > 0:
        resting.append(incoming)
    return trades

# Micro-benchmark: 5,000 resting + 5,000 incoming
random.seed(0)
book = [Order(id=i, side="sell", price=100 + random.randint(0, 20), qty=10)
        for i in range(5_000)]
start = time.perf_counter()
for i in range(5_000):
    match_bad(book, Order(id=100_000+i, side="buy", price=110, qty=1))
print(f"BAD matcher: {time.perf_counter()-start:.3f}s for 10k events")

### Best version: per-price FIFO queues

Two dicts `bids[price]` and `asks[price]`, each mapping to a `deque` of orders in arrival order. The best price on each side is the dict's min/max key. Matching pops from the front of a deque — O(1) — and we only ever touch the *best* price level.

We also support **cancel** and **partial fills** correctly.

In [ ]:
from collections import defaultdict, deque

class MatchingEngine:
    """Price-time priority limit order book.

    Two invariants carry the whole design:
      1. PRICE priority — we always take the best opposing price level first.
      2. TIME priority  — within one price level, FIFO. Oldest order fills first.
    Both are proved by the test cell below, not just asserted in prose.
    """

    def __init__(self):
        self.bids: dict[int, deque[Order]] = defaultdict(deque)   # price → FIFO
        self.asks: dict[int, deque[Order]] = defaultdict(deque)
        self.index: dict[int, Order] = {}                         # order id → Order (for cancel)

    # ----- helpers ---------------------------------------------------
    def _best_ask(self): return min(self.asks) if self.asks else None
    def _best_bid(self): return max(self.bids) if self.bids else None

    def _opposite(self, side): return self.asks if side == "buy" else self.bids

    def _crosses(self, o: Order, level: int) -> bool:
        """Would an order `o` trade against a resting order at price `level`?"""
        if o.otype == "market":
            return True                      # market orders accept any price
        return level <= o.price if o.side == "buy" else level >= o.price

    def _levels_best_first(self, side):
        book = self._opposite(side)
        return sorted(book) if side == "buy" else sorted(book, reverse=True)

    def _rest(self, o):
        (self.bids if o.side == "buy" else self.asks)[o.price].append(o)
        self.index[o.id] = o

    # ----- public API ------------------------------------------------
    def fillable(self, o: Order) -> int:
        """How much of `o` *could* fill right now. Read-only — mutates nothing.
        Needed by fill-or-kill, which must decide before it touches the book."""
        book, got = self._opposite(o.side), 0
        for level in self._levels_best_first(o.side):
            if not self._crosses(o, level):
                break
            for resting in book[level]:
                got += resting.qty
                if got >= o.qty:
                    return o.qty
        return got

    def submit(self, o: Order):
        # Fill-or-kill is all-or-nothing: check *before* mutating anything,
        # otherwise a partial sweep would leave the book in a half-eaten state.
        if o.otype == "fok" and self.fillable(o) < o.qty:
            return []

        trades = []
        book = self._opposite(o.side)
        while o.qty and book:
            best = self._best_ask() if o.side == "buy" else self._best_bid()
            if not self._crosses(o, best):
                break
            q = book[best]
            top = q[0]                                  # ← TIME priority: front of queue
            fill = min(top.qty, o.qty)
            # The trade prints at the RESTING order's price. Any price improvement
            # belongs to whoever was there first — that is what "time priority" pays for.
            trades.append((o.id, top.id, best, fill) if o.side == "buy"
                          else (top.id, o.id, best, fill))
            top.qty -= fill
            o.qty  -= fill
            if top.qty == 0:
                q.popleft(); self.index.pop(top.id, None)
                if not q: del book[best]

        # Only a LIMIT order leaves a remainder resting on the book.
        # market / ioc / fok all discard whatever they could not fill.
        if o.qty and o.otype == "limit":
            self._rest(o)
        return trades

    def cancel(self, order_id: int) -> bool:
        o = self.index.pop(order_id, None)
        if o is None:
            return False
        side_book = self.bids if o.side == "buy" else self.asks
        side_book[o.price].remove(o)           # O(n) within one price level; usually tiny
        if not side_book[o.price]:
            del side_book[o.price]
        return True

    def top_of_book(self):
        return self._best_bid(), self._best_ask()

# --- the original smoke test, still green ----------------------------
e = MatchingEngine()
assert e.submit(Order(1, "sell", 101, 10)) == []
assert e.submit(Order(2, "sell", 102,  5)) == []
assert e.submit(Order(3, "buy", 101, 8)) == [(3, 1, 101, 8)]         # cross one level
trades = e.submit(Order(4, "buy", 103, 20))                          # sweep two, rest the rest
assert trades == [(4, 1, 101, 2), (4, 2, 102, 5)], trades
assert e.top_of_book() == (103, None)
assert e.cancel(4) is True
assert e.top_of_book() == (None, None)
assert e.cancel(4) is False     # already cancelled → idempotent False
print("✅ matcher: smoke test passed")

# ---- benchmark against the bad version -----------------------------
import time
e2 = MatchingEngine()
for i in range(5_000):
    e2.submit(Order(i, "sell", 100 + random.randint(0,20), 10))
start = time.perf_counter()
for i in range(5_000):
    e2.submit(Order(100_000+i, "buy", 110, 1))
print(f"BEST matcher: {time.perf_counter()-start:.3f}s for 10k events")

> 🚀 Real exchanges go further — `SortedDict` / skip lists for O(log n) price lookup, pre-allocated intrusive linked-list nodes to avoid GC pressure, etc. But the **shape** is exactly what we wrote above.

### Proving price-time priority

"The matching engine works" is not a claim you get to make casually — a
priority bug is *silent*, it just quietly gives the wrong trader the fill.
So we test the two invariants directly, plus the two ways an order can
partially fill.

Read the assertions before running them and predict each answer. If your
prediction and the assert disagree, one of you has a bug.

In [ ]:
# ===== 1. TIME priority: same price, first-in fills first =================
e = MatchingEngine()
e.submit(Order(101, "sell", 100, 5))      # arrived FIRST
e.submit(Order(102, "sell", 100, 5))      # same price, arrived second
assert e.submit(Order(103, "buy", 100, 5)) == [(103, 101, 100, 5)]   # 101, not 102
assert e.submit(Order(104, "buy", 100, 5)) == [(104, 102, 100, 5)]   # then 102

# ===== 2. PRICE priority outranks time ===================================
e = MatchingEngine()
e.submit(Order(201, "sell", 102, 5))      # arrived FIRST but is the worse price
e.submit(Order(202, "sell", 100, 5))      # arrived second, better price
assert e.submit(Order(203, "buy", 102, 5)) == [(203, 202, 100, 5)]   # cheaper wins

# ===== 3. Price improvement accrues to the RESTING order =================
# Buyer offers 105; the only ask left is at 102 → the trade prints at 102,
# and the *seller* keeps the 3-tick improvement, not the buyer.
assert e.submit(Order(204, "buy", 105, 5)) == [(204, 201, 102, 5)]

# ===== 4. Partial fill of the RESTING order — it keeps its queue slot ====
e = MatchingEngine()
e.submit(Order(301, "sell", 100, 10))
assert e.submit(Order(302, "buy", 100, 4)) == [(302, 301, 100, 4)]
assert e.index[301].qty == 6                  # shrank, but still at the front
assert e.submit(Order(303, "buy", 100, 4)) == [(303, 301, 100, 4)]
assert e.index[301].qty == 2

# ===== 5. Partial fill of the INCOMING order — it sweeps, then rests =====
e = MatchingEngine()
e.submit(Order(401, "sell", 100, 3))
e.submit(Order(402, "sell", 101, 3))
assert e.submit(Order(403, "buy", 101, 10)) == [(403, 401, 100, 3), (403, 402, 101, 3)]
assert e.index[403].qty == 4                  # 10 - 3 - 3 left over
assert e.top_of_book() == (101, None)         # …resting as the new best bid

# ===== 6. Mirror image: an aggressive SELL walks the bid side ============
e = MatchingEngine()
e.submit(Order(501, "buy", 100, 5))
e.submit(Order(502, "buy", 100, 5))
assert e.submit(Order(503, "sell", 100, 7)) == [(501, 503, 100, 5), (502, 503, 100, 2)]
assert e.index[502].qty == 3

# ===== 7. A non-crossing order just rests; it never trades ===============
e = MatchingEngine()
e.submit(Order(601, "sell", 105, 5))
assert e.submit(Order(602, "buy", 100, 5)) == []
assert e.top_of_book() == (100, 105)          # a 5-tick spread, no trade

print("✅ price-time priority: 7/7 invariants hold")

## 2. Order types — what happens to the remainder?

Every order type answers exactly one question: **what do we do with the
quantity that could not be filled right now?**

| Type | Price limit | Unfilled remainder | Typical use |
|---|---|---|---|
| `limit`  | yes | **rests** on the book | "I'll wait for my price" — this is what makes a market |
| `market` | none — takes any price | **discarded** | "get me in now" — pays the spread, risks a bad sweep |
| `ioc`    | yes | **discarded** | probe for liquidity without leaving a footprint |
| `fok`    | yes | **nothing trades at all** | multi-leg strategies where a partial fill is worse than none |

Note the honest costs. A `market` order into a thin book sweeps every level
and can print an appalling price (this is how "flash crash" fills happen) —
which is why most venues actually implement it as a limit order pegged far
away, plus a price collar. `fok` needs a read-only pre-scan of the book, so
it is the one order type that is *not* O(levels touched).

In [ ]:
# ===== MARKET: takes any price, never rests ==============================
e = MatchingEngine()
e.submit(Order(701, "sell", 100, 2))
e.submit(Order(702, "sell", 500, 2))      # an awful price, far from the touch
tr = e.submit(Order(703, "buy", 0, 10, otype="market"))
assert tr == [(703, 701, 100, 2), (703, 702, 500, 2)]   # ← swept into the 500 level!
assert e.top_of_book() == (None, None)    # the 6 unfilled units simply evaporate
print("market : filled 4 of 10, average price",
      f"{sum(p*q for _,_,p,q in tr)/sum(q for *_ ,q in tr):.0f} ticks",
      "— note it paid 500 for half of it")

# ===== MARKET into an empty book: dies quietly ===========================
e = MatchingEngine()
assert e.submit(Order(704, "buy", 0, 5, otype="market")) == []
assert e.top_of_book() == (None, None)

# ===== IOC: fill what you can at your limit, cancel the rest =============
e = MatchingEngine()
e.submit(Order(801, "sell", 100, 2))
e.submit(Order(802, "sell", 105, 5))
assert e.submit(Order(803, "buy", 100, 10, otype="ioc")) == [(803, 801, 100, 2)]
assert e.top_of_book() == (None, 105)     # the 8 unfilled units did NOT rest
print("ioc    : took the 2 available at 100, left no footprint on the book")

# ===== FOK: all-or-nothing, and it must not disturb the book on reject ===
e = MatchingEngine()
e.submit(Order(901, "sell", 100, 2))
assert e.submit(Order(902, "buy", 100, 10, otype="fok")) == []   # only 2 avail → reject
assert e.index[901].qty == 2              # ← the crucial bit: book is untouched
assert e.submit(Order(903, "buy", 100, 2, otype="fok")) == [(903, 901, 100, 2)]
assert e.top_of_book() == (None, None)
print("fok    : rejected 10 without touching the book, then filled 2 cleanly")

# ===== FOK across several levels — fillable() must look past the touch ===
e = MatchingEngine()
e.submit(Order(910, "sell", 100, 3))
e.submit(Order(911, "sell", 101, 3))
assert e.submit(Order(912, "buy", 101, 6, otype="fok")) == \
       [(912, 910, 100, 3), (912, 911, 101, 3)]
assert e.top_of_book() == (None, None)

# ===== A limit order is the only one that rests ==========================
e = MatchingEngine()
for n, t in enumerate(("market", "ioc", "fok"), start=990):
    e.submit(Order(n, "buy", 100, 5, otype=t))
assert e.top_of_book() == (None, None), "non-limit orders must never rest"
e.submit(Order(999, "buy", 100, 5, otype="limit"))
assert e.top_of_book() == (100, None)

print("\n✅ order types: market / ioc / fok / limit all behave")

## 3. Determinism & WAL replay

Every input message is **written to an append-only file before it is matched**. After a crash we replay the file into a fresh matcher and recover the exact same state, trade for trade. This is the single most important correctness property of any exchange.

In [ ]:
import io, json

# An in-memory WAL to keep the notebook self-contained (a real one is fsync'd to disk).
wal = io.StringIO()

def journal_and_submit(engine, order: Order):
    wal.write(json.dumps(order.__dict__) + "\n")
    # a real implementation would fsync() here before matching
    return engine.submit(order)

# --- live run --------------------------------------------------------
live = MatchingEngine()
inputs = [
    Order(1, "sell", 101, 10), Order(2, "sell", 102, 5),
    Order(3, "buy",  101, 8),  Order(4, "buy",  103, 20),
]
live_trades = [journal_and_submit(live, o) for o in inputs]

# --- crash! replay the WAL into a fresh engine -----------------------
replayed = MatchingEngine()
wal.seek(0)
replay_trades = [replayed.submit(Order(**json.loads(line))) for line in wal]

assert live_trades == replay_trades, "non-deterministic! 🔥"
assert live.top_of_book() == replayed.top_of_book()
print("✅ WAL replay reproduced the book exactly — identical trades")
print("   top of book:", replayed.top_of_book())

## 4. Market-data fan-out

One trade can be interesting to **hundreds of thousands** of subscribers. If the matcher itself writes to each subscriber socket, one slow consumer back-pressures the whole exchange — every trader pays for it.

### Bad: matcher pushes directly to subscribers

In [ ]:
import time

class SlowSubscriber:
    def __init__(self): self.received = []
    def on_trade(self, t):
        time.sleep(0.002)           # 2 ms of 'network' per message
        self.received.append(t)

bad_subs = [SlowSubscriber() for _ in range(5)]

def matcher_bad(trades):
    start = time.perf_counter()
    for t in trades:
        for s in bad_subs:          # ❌ matcher is blocked on every subscriber
            s.on_trade(t)
    return time.perf_counter() - start

t = matcher_bad([f"trade-{i}" for i in range(50)])
print(f"BAD fan-out: matcher hot-path spent {t*1000:6.1f} ms delivering to subs")

### Best: matcher only puts onto a queue; a publisher thread fans out

In [ ]:
import queue, threading

good_subs = [SlowSubscriber() for _ in range(5)]
feed: queue.Queue = queue.Queue()

def publisher():
    while True:
        msg = feed.get()
        if msg is None: return
        for s in good_subs:
            s.on_trade(msg)

pub = threading.Thread(target=publisher, daemon=True); pub.start()

def matcher_best(trades):
    start = time.perf_counter()
    for t in trades:
        feed.put(t)                 # ✅ O(1) enqueue, no blocking
    return time.perf_counter() - start

t = matcher_best([f"trade-{i}" for i in range(50)])
print(f"BEST fan-out: matcher hot-path spent {t*1000:6.1f} ms enqueuing")

feed.put(None); pub.join()
assert len(good_subs[0].received) == 50
print("   all subscribers received every trade — work happened off the hot path")

## Where to go next

- **Self-match prevention** — if the incoming trader would match their own
  resting order, cancel one of them (venues differ on *which*). Cheap to add,
  and required by most regulators — but note it breaks strict FIFO: the order
  behind yours in the queue jumps ahead of it.
- **Iceberg orders** — show 100 shares, hide 10,000. Each refill goes to the
  **back** of the queue, which is the whole point: you get hidden size, and
  you pay for it in time priority.
- **Circuit breakers** — halt trading for a symbol if price moves >X% in Y
  seconds; show halted status on the feed.
- **Snapshot + delta market data** — send a full book snapshot once, then only
  the deltas. Subscribers that fall behind re-sync from the next snapshot.
- **Sharding** — one matcher per symbol, spread across machines; the gateway
  routes by symbol. Cross-symbol atomicity (spread trades, baskets) is then
  *not* available without a distributed transaction — a real cost of this shape.

### Further reading

- **LMAX Disruptor paper** — single-threaded design that hits millions of ops/s on one core.
- **Nasdaq ITCH 5.0 spec** — the actual wire format of a real market-data feed.
- **CME MDP 3.0** — how one of the world's largest exchanges publishes market data.
- **`04-patterns/`** in this repo — WAL, leader election, and consistent hashing are all used in production matching engines.